# Семинар 5. Регрессия и классификация для бизнес-прогнозов

**Цель семинара:** Научиться методологии сборки финальной аналитической витрины данных (Analytical Base Table - ABT). Разделить данные, обучить классические baseline-модели (Логистическая регрессия, Дерево решений), оценить их бизнес-метрики и провести пакетный инференс (batch inference) для восстановления утерянных целевых меток. Итогом станут две production-функции для `src/data_pipeline.py` и `src/model_training.py`.

### 🔧 Настройка окружения и импорт библиотек


In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)


---

## 📥 Шаг 1. Инициализация локального контура (Сборка витрины АВТ)

Модель машинного обучения не может обучаться на трех разных таблицах. Нам необходимо выполнить Data Orchestration — сквозное объединение очищенного профиля, RFM-кластеров и NLP-метрик по ключу `Target_ID`.


In [ ]:
INPUT_DIR = os.path.join(".", "data", "processed")
os.makedirs(INPUT_DIR, exist_ok=True)

# Генерация mock-датасетов для симуляции слияния (предыдущие семинары)
mock_profiles = pd.DataFrame({
    'Target_ID': ['T01', 'T02', 'T03', 'T04', 'T05'],
    'Target_Flag': [1, 0, np.nan, 0, 1], # T03 имеет утерянный таргет (для инференса)
    'MonthlyCharges': [50.0, 20.0, 100.0, 105.0, 45.5],
    'gender_male': [1, 0, 1, 1, 0] # Уже закодировано OHE
})

mock_rfm = pd.DataFrame({
    'Target_ID': ['T01', 'T02', 'T04', 'T05'], # T03 не совершал транзакций
    'Recency': [5, 100, 2, 10],
    'Monetary': [500, 50, 1000, 200],
    'Cluster_ID': [1, 0, 2, 1]
})

mock_nlp = pd.DataFrame({
    'Target_ID': ['T01', 'T03', 'T05'],
    'Mean_Sentiment': [-0.8, 0.5, 0.9]
})

print("Имитация входящих таблиц успешно загружена.")


---

## 🛠 ЗАДАНИЕ 1: Сборка аналитической витрины (Data Orchestration)
**Бизнес-контекст:** Мы объединяем данные через Left Join, где левой (главной) таблицей выступает мастер-профиль. Если клиент не совершал транзакций (нет в RFM) или не писал отзывы (нет в NLP), после джойна образуются пропуски (NaN). Их нужно безопасно заполнить (например, нулями для Monetary и нейтральным 0.0 для Mean_Sentiment).

**Инструкция (TODO):**
1. Слейте df_profiles и df_rfm по колонке Target_ID (используйте pd.merge(..., how='left')).
2. Слейте результат с df_nlp.
3. Заполните образовавшиеся NaN в метриках RFM нулями, а в Mean_Sentiment значением 0.0.

*🤖 Теги для AI-ментора: #SEM5_TASK1_START, #SEM5_TASK1_BUG*


In [ ]:
# TODO: 1.1. Выполните left join mock_profiles и mock_rfm по 'Target_ID'
# TODO: 1.2. К результату примените left join с mock_nlp
# TODO: 1.3. Заполните пропуски через .fillna()
raise NotImplementedError("Задание 1 не выполнено! Удалите эту строку и напишите свой код.")

df_abt = pd.merge(..., ..., on='Target_ID', how='left')
df_abt = pd.merge(..., ..., on='Target_ID', how='left')

# Заполнение пропусков
# Ваш код...

display(df_abt.head())


---

## 🛠 ЗАДАНИЕ 2: Изоляция пула инференса и разбиение выборки
**Бизнес-контекст:** В нашей витрине есть строки, где целевая переменная (Target_Flag / Churn) равна NaN. Это новые клиенты, чей статус мы хотим предсказать. Их нужно отделить в df_inference. Оставшиеся размеченные данные (df_historical) мы разобьем на обучающую (train) и тестовую (test) выборки для тренировки модели.

**Инструкция (TODO):**
1. Изолируйте строки без таргета: df_inference = df_abt[df_abt['Target_Flag'].isna()].
2. Изолируйте исторические данные: df_historical = df_abt[df_abt['Target_Flag'].notna()].
3. Разделите df_historical на X (фичи) и y (таргет). Уберите Target_ID из X.
4. Используйте train_test_split(X, y, test_size=0.2, random_state=42) для разбиения.

*🤖 Теги для AI-ментора: #SEM5_TASK2_START, #SEM5_TASK2_BUG*


In [ ]:
# TODO: 2.1. Отфильтруйте df_abt на df_inference (isna) и df_historical (notna)
# TODO: 2.2. Разбейте df_historical на X (без Target_ID и Target_Flag) и y
# TODO: 2.3. Примените train_test_split
raise NotImplementedError("Задание 2 не выполнено! Удалите эту строку и напишите свой код.")

df_inference = df_abt[df_abt['Target_Flag'].isna()].copy()
df_historical = df_abt[df_abt['Target_Flag'].notna()].copy()

X = df_historical.drop(columns=[...])
y = df_historical[...]

X_train, X_test, y_train, y_test = train_test_split(..., ..., test_size=0.2, random_state=42)


---

## 🛠 ЗАДАНИЕ 3: Обучение Baseline-моделей и «Парадокс точности»
**Бизнес-контекст:** Дисбаланс классов (например, 90% остаются, 10% уходят) делает метрику Accuracy (доля правильных ответов) бесполезной. Модель, говорящая "никто не уйдет", получит Accuracy 90%, но принесет бизнесу 0 пользы. Поэтому мы смотрим на Precision (точность) и Recall (полноту).

**Инструкция (TODO):**
1. Инициализируйте LogisticRegression(random_state=42) и DecisionTreeClassifier(max_depth=3, random_state=42).
2. Обучите их на X_train, y_train.
3. Сделайте предсказания на X_test и посчитайте метрики precision_score и recall_score.

*🤖 Теги для AI-ментора: #SEM5_TASK3_START, #SEM5_TASK3_WHY*


In [ ]:
# TODO: 3.1. Создайте объекты моделей с random_state=42
# TODO: 3.2. Вызовите метод .fit() для X_train, y_train
# TODO: 3.3. Вызовите метод .predict() для X_test
raise NotImplementedError("Задание 3 не выполнено! Удалите эту строку и напишите свой код.")

lr_model = LogisticRegression(random_state=42)
dt_model = ...

lr_model.fit(..., ...)
dt_model.fit(..., ...)

y_pred_lr = lr_model.predict(...)
y_pred_dt = dt_model.predict(...)


---

## 🛠 ЗАДАНИЕ 4: Экономический аудит матрицы ошибок (Confusion Matrix)
**Бизнес-контекст:** * False Positive (Ошибка I рода): Ложная тревога. Мы дали клиенту скидку, а он и так не собирался уходить. (Потеря маржи).
* False Negative (Ошибка II рода): Пропуск события. Клиент ушел, а мы не заметили. (Потеря LTV клиента).
В большинстве бизнес-доменов False Negative обходится гораздо дороже!

**Инструкция (TODO):**
1. Рассчитайте confusion_matrix(y_test, y_pred_lr).
2. Визуализируйте её через sns.heatmap.


In [ ]:
# TODO: 4.1. Рассчитайте confusion_matrix
# TODO: 4.2. Постройте sns.heatmap
raise NotImplementedError("Задание 4 не выполнено! Удалите эту строку и напишите свой код.")

cm = confusion_matrix(..., ...)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Матрица ошибок')
plt.show()


---

## 🏗 ФИНАЛЬНАЯ СБОРКА: Функции для модулей ядра

В src/data_pipeline.py мы добавим функцию сборки витрины assemble_abt. 
В src/model_training.py — функцию обучения базовых моделей train_baseline_models.


In [ ]:
def assemble_abt(dfs_list: list, on_col: str) -> pd.DataFrame:
    # TODO: Реализуйте последовательный pd.merge(how='left') для списка датафреймов
    raise NotImplementedError("Функция assemble_abt не готова!")

def train_baseline_models(df: pd.DataFrame, target_col: str, feature_cols: list, task_type: str = 'classification') -> dict:
    # TODO: Реализуйте сепарацию, train_test_split, обучение LogisticRegression, подсчет метрик и инференс на NaN.
    raise NotImplementedError("Функция train_baseline_models не готова!")


---

## 🛠 Автоматизированная проверка качества (Autocheck)

Скрипт проверяет сборку АВТ и корректность возвращаемого словаря модели.


In [ ]:
def run_autocheck(abt_df, model_result_dict):
    print("🚀 Тестирование сборки АВТ и Baseline пайплайна...\n" + "-"*45)
    validation_status = True
    
    if not isinstance(abt_df, pd.DataFrame):
        print("❌ Ошибка: assemble_abt не вернула DataFrame.")
        validation_status = False
    elif abt_df.isnull().sum().sum() > 0 and 'Target_Flag' not in abt_df.columns[abt_df.isnull().any()]:
        print("❌ Ошибка: В собранной витрине остались NaN в фичах. Заполните их нулями.")
        validation_status = False
    else:
        print("✅ Витрина АВТ успешно собрана, пропуски в фичах изолированы.")
        
    expected_keys = ['model', 'metrics', 'inference_shape']
    if not isinstance(model_result_dict, dict) or not all(k in model_result_dict for k in expected_keys):
        print(f"❌ Ошибка: Словарь результата должен содержать ключи: {expected_keys}")
        validation_status = False
    else:
        print("✅ Функция обучения успешно вернула обученную модель и метрики.")
        
    if os.path.exists(os.path.join(INPUT_DIR, 'recovered_targets.csv')):
         print("✅ Пакетный инференс выполнен, файл recovered_targets.csv сохранен.")
    else:
         print("❌ Ошибка: Файл пакетного инференса recovered_targets.csv не найден.")
         validation_status = False

    print("-" * 45)
    if validation_status:
        print("🎉 ПОЗДРАВЛЯЕМ! Пайплайн обучения и инференса готов.")
        print("Разнесите эти функции в course_project/src/data_pipeline.py и model_training.py!")
    else:
        print("⚠️ Обнаружены дефекты.")

run_autocheck(df_abt_final, result)
